# Дипломный проект

Рекомендательная система для интернет-магазина.

Данные:

- `events.csv.zip` — лог событий пользователей;
- `category_tree.csv` — дерево категорий товаров;
- `item_properties_part1.csv`, `item_properties_part2.csv` — свойства товаров.

## 1. Импорт библиотек и загрузка данных

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
DATA_DIR = Path("data")

events_path = DATA_DIR / "events.csv.zip"
category_tree_path = DATA_DIR / "category_tree.csv"
item_properties_part1_path = DATA_DIR / "item_properties_part1.csv"
item_properties_part2_path = DATA_DIR / "item_properties_part2.csv"

for path in [events_path, category_tree_path, item_properties_part1_path, item_properties_part2_path]:
    print(path, "->", path.exists())

data\events.csv.zip -> True
data\category_tree.csv -> True
data\item_properties_part1.csv -> True
data\item_properties_part2.csv -> True


In [3]:
events = pd.read_csv(events_path)

print(f"Размер events: {events.shape[0]:,} строк и {events.shape[1]} столбцов")
events.head()

Размер events: 2,756,101 строк и 5 столбцов


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


Для удобства переведем `timestamp` из Unix time в миллисекундах в обычную дату и время.

In [4]:
events["event_time"] = pd.to_datetime(events["timestamp"], unit="ms")

print("Минимальная дата:", events["event_time"].min())
print("Максимальная дата:", events["event_time"].max())

events.head()

Минимальная дата: 2015-05-03 03:00:04.384000
Максимальная дата: 2015-09-18 02:59:47.788000


,timestamp,visitorid,event,itemid,transactionid,event_time
0,1433221332117,257597,view,355908,NaN,2015-06-02 05:02:12.117
1,1433224214164,992329,view,248676,NaN,2015-06-02 05:50:14.164
2,1433221999827,111016,view,318965,NaN,2015-06-02 05:13:19.827
3,1433221955914,483717,view,253185,NaN,2015-06-02 05:12:35.914
4,1433221337106,951259,view,367447,NaN,2015-06-02 05:02:17.106


Посмотрим на дополнительные таблицы, чтобы понимать, какие еще данные есть в проекте.

In [5]:
category_tree = pd.read_csv(category_tree_path)

print(f"Размер category_tree: {category_tree.shape[0]:,} строк и {category_tree.shape[1]} столбцов")
category_tree.head()

Размер category_tree: 1,669 строк и 2 столбцов


,categoryid,parentid
0,1016,213.00
1,809,169.00
2,570,9.00
3,1691,885.00
4,536,1691.00


In [6]:
item_properties_sample = pd.concat(
    [
        pd.read_csv(item_properties_part1_path, nrows=5),
        pd.read_csv(item_properties_part2_path, nrows=5),
    ],
    ignore_index=True,
)

item_properties_sample

,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1441508400000,206783,888,1116713 960601 n277.200
2,1439089200000,395014,400,n552.000 639502 n720.000 424566
3,1431226800000,59481,790,n15360.000
4,1431831600000,156781,917,828513
5,1433041200000,183478,561,769062
6,1439694000000,132256,976,n26.400 1135780
7,1435460400000,420307,921,1149317 1257525
8,1431831600000,403324,917,1204143
9,1435460400000,230701,521,769062


## Задание 1.1. Сколько записей событий находится в датасете?

In [7]:
events_count = len(events)

print(f"Количество записей событий в датасете: {events_count:,}")

Количество записей событий в датасете: 2,756,101


**Ответ:** в датасете `events` находится **2 756 101** запись событий.

## Задание 1.2. Какие типы событий содержатся в датасете?

In [8]:
event_counts = events["event"].value_counts()
event_shares = events["event"].value_counts(normalize=True).mul(100).round(2)

event_summary = pd.DataFrame({
    "count": event_counts,
    "share_percent": event_shares,
})

event_summary

,count,share_percent
event,,
view,2664312,96.67
addtocart,69332,2.52
transaction,22457,0.81


**Ответ:** в датасете есть три типа событий:

- `view` — просмотр товара;
- `addtocart` — добавление товара в корзину;
- `transaction` — покупка / транзакция.

## Задание 1.3. Какой процент продаж обеспечивают топовые товары?

Условие: взять статистику **до 1 июля включительно**, найти топ-3 товаров по числу транзакций, а затем проверить, какую долю покупок эти товары покрывают **после 1 июля**.

Так как в данных даты относятся к 2015 году, граница для обучения: `2015-07-01 23:59:59.999`. Все покупки позже этой даты считаем будущим периодом.

In [9]:
transactions = events[events["event"] == "transaction"].copy()

print(f"Количество строк с покупками: {len(transactions):,}")
print(f"Количество уникальных transactionid: {transactions['transactionid'].nunique():,}")
transactions.head()

Количество строк с покупками: 22,457
Количество уникальных transactionid: 17,672


,timestamp,visitorid,event,itemid,transactionid,event_time
130,1433222276276,599528,transaction,356475,4000.00,2015-06-02 05:17:56.276
304,1433193500981,121688,transaction,15335,11117.00,2015-06-01 21:18:20.981
418,1433193915008,552148,transaction,81345,5444.00,2015-06-01 21:25:15.008
814,1433176736375,102019,transaction,150318,13556.00,2015-06-01 16:38:56.375
843,1433174518180,189384,transaction,310791,7244.00,2015-06-01 16:01:58.180


In [10]:
cutoff = pd.Timestamp("2015-07-01 23:59:59.999")

before_july = transactions[transactions["event_time"] <= cutoff]
after_july = transactions[transactions["event_time"] > cutoff]

print("Покупок до 1 июля включительно:", f"{len(before_july):,}")
print("Покупок после 1 июля:", f"{len(after_july):,}")
print("Последняя покупка до/на 1 июля:", before_july["event_time"].max())
print("Первая покупка после 1 июля:", after_july["event_time"].min())

Покупок до 1 июля включительно: 9,849
Покупок после 1 июля: 12,608
Последняя покупка до/на 1 июля: 2015-07-01 23:57:26.604000
Первая покупка после 1 июля: 2015-07-02 00:02:21.950000


In [11]:
top_3_items = (
    before_july
    .groupby("itemid")
    .size()
    .sort_values(ascending=False)
    .head(3)
)

top_3_items_df = top_3_items.reset_index(name="transactions_before_july")
top_3_items_df

,itemid,transactions_before_july
0,119736,36
1,369447,31
2,7943,30


In [12]:
top_3_item_ids = top_3_items.index

covered_purchases = after_july["itemid"].isin(top_3_item_ids).sum()
total_future_purchases = len(after_july)
coverage_percent = covered_purchases / total_future_purchases * 100

print("Топ-3 товаров:", list(top_3_item_ids))
print("Покупок после 1 июля всего:", f"{total_future_purchases:,}")
print("Покупок после 1 июля среди топ-3 товаров:", f"{covered_purchases:,}")
print(f"Доля покрытых покупок: {coverage_percent:.2f}%")

Топ-3 товаров: [119736, 369447, 7943]
Покупок после 1 июля всего: 12,608
Покупок после 1 июля среди топ-3 товаров: 77
Доля покрытых покупок: 0.61%


**Ответ:** топ-3 товара по покупкам до 1 июля включительно: `119736`, `369447`, `7943`.

После 1 июля эти товары были куплены **77** раз из **12 608** покупок. Значит, они покрывают примерно **0.61 %** последующих покупок.

Вывод: простая рекомендация только самых популярных товаров почти не покрывает будущие покупки. Для бизнес-цели по росту допродаж лучше строить персональные или хотя бы сегментные рекомендации.

## Задание 1.4. Какой тип валидации нужно использовать?

Для рекомендательной системы важно проверять модель так, как она будет работать в реальности: обучаться на прошлых событиях и предсказывать будущие покупки пользователя.

Поэтому случайное разбиение или стратификация могут привести к утечке информации из будущего в обучение.

**Ответ:** нужно использовать **разбиение по времени**.

Например:

- train — события до выбранной даты;
- validation/test — события после выбранной даты;
- качество считаем через `Precision@3`, так как сервис должен выдавать три рекомендации пользователю.

## Итоговые ответы

1. В датасете `events` находится **2 756 101** запись.
2. Типы событий: **`view`**, **`addtocart`**, **`transaction`**.
3. Топ-3 товара по покупкам до 1 июля включительно покрывают **0.61 %** покупок после 1 июля.
4. Для валидации нужно использовать **разбиение по времени**.

# 2. факторы, эксперименты и сервис

## Задание 2.1. Какое свойство айтемов не входит в топ-20?

В `FeatureGenerationNotebook.ipynb` есть идея использовать самые частые свойства товаров. Сергей на 3-й неделе говорил про топ-20, но в коде был срез `[:10]`. Ниже посчитаем настоящий топ-20 по числу уникальных товаров, у которых встречалось свойство.

Важно убрать дубли `itemid-property`, потому что свойство одного товара могло записываться несколько раз в разные моменты времени.

In [13]:
property_frames = [
    pd.read_csv(item_properties_part1_path, usecols=["itemid", "property"]),
    pd.read_csv(item_properties_part2_path, usecols=["itemid", "property"]),
]

properties_for_top = pd.concat(property_frames, ignore_index=True)
properties_for_top = properties_for_top.drop_duplicates(["itemid", "property"])

property_top = (
    properties_for_top
    .groupby("property")["itemid"]
    .nunique()
    .sort_values(ascending=False)
)

property_top.head(25).reset_index(name="unique_items")

,property,unique_items
0,categoryid,417053
1,159,417053
2,790,417053
3,364,417053
4,112,417053
5,888,417053
6,764,417053
7,283,417053
8,available,417053
9,678,417019


**Ответ:** в топ-20 входят свойства 888, 790, и categoryid. Из представленных вариантов ответов **color** не входит в топ-20.

## Задание 2.2. Эксперименты с моделями рекомендаций

Для проверки используем временное разбиение, потому что в реальном сервисе модель учится на прошлом и рекомендует товары для будущих покупок.

- train: события до `2015-07-01 23:59:59.999`;
- test: события после этой даты;
- целевое действие: `transaction`;
- метрика: `Precision@3`;
- одна рекомендация сервиса — список из трех `itemid`.

In [14]:
EVENT_WEIGHTS = {
    "view": 1,
    "addtocart": 3,
    "transaction": 10,
}


def precision_at_k(relevant_by_user, recommendations_by_user, fallback, k=3):
    scores = []

    for visitor_id, relevant_items in relevant_by_user.items():
        recommendations = []
        seen = set()

        for item_id in recommendations_by_user.get(visitor_id, []) + fallback:
            if item_id not in seen:
                recommendations.append(item_id)
                seen.add(item_id)
            if len(recommendations) == k:
                break

        scores.append(len(set(recommendations) & relevant_items) / k)

    return sum(scores) / len(scores) if scores else 0


def build_weighted_recommendations(events_part):
    weighted = events_part.copy()
    weighted["weight"] = weighted["event"].map(EVENT_WEIGHTS).fillna(0).astype(int)

    fallback = (
        weighted.groupby("itemid")["weight"]
        .sum()
        .sort_values(ascending=False)
        .head(200)
        .index
        .astype(int)
        .tolist()
    )

    user_item_scores = (
        weighted.groupby(["visitorid", "itemid"])["weight"]
        .sum()
        .reset_index()
        .sort_values(["visitorid", "weight"], ascending=[True, False])
    )

    recommendations_by_user = {
        int(visitor_id): group["itemid"].astype(int).head(20).tolist()
        for visitor_id, group in user_item_scores.groupby("visitorid", sort=False)
    }

    return recommendations_by_user, fallback

In [15]:
cutoff = pd.Timestamp("2015-07-01 23:59:59.999")

train_events = events[events["event_time"] <= cutoff].copy()
test_events = events[events["event_time"] > cutoff].copy()
test_transactions = test_events[test_events["event"] == "transaction"]

relevant_by_user = (
    test_transactions
    .groupby("visitorid")["itemid"]
    .agg(lambda values: set(values.astype(int)))
    .to_dict()
)

print("Пользователей с покупками в test:", len(relevant_by_user))

Пользователей с покупками в test: 6531


In [16]:
transaction_fallback = (
    train_events[train_events["event"] == "transaction"]["itemid"]
    .value_counts()
    .head(200)
    .index
    .astype(int)
    .tolist()
)

view_fallback = (
    train_events["itemid"]
    .value_counts()
    .head(200)
    .index
    .astype(int)
    .tolist()
)

weighted_recs, weighted_fallback = build_weighted_recommendations(train_events)

strong_events = train_events[train_events["event"].isin(["addtocart", "transaction"])]
strong_recs, _ = build_weighted_recommendations(strong_events)

experiment_results = pd.DataFrame(
    [
        {
            "model": "popular_transactions",
            "description": "Топ товаров по покупкам в train",
            "precision_at_3": precision_at_k(relevant_by_user, {}, transaction_fallback),
        },
        {
            "model": "popular_views",
            "description": "Топ товаров по просмотрам и всем событиям",
            "precision_at_3": precision_at_k(relevant_by_user, {}, view_fallback),
        },
        {
            "model": "weighted_popularity",
            "description": "Глобальная популярность с весами событий",
            "precision_at_3": precision_at_k(relevant_by_user, {}, weighted_fallback),
        },
        {
            "model": "personal_history_weighted",
            "description": "Персональная история пользователя с весами событий",
            "precision_at_3": precision_at_k(relevant_by_user, weighted_recs, weighted_fallback),
        },
        {
            "model": "strong_history",
            "description": "История пользователя только по корзинам и покупкам",
            "precision_at_3": precision_at_k(relevant_by_user, strong_recs, weighted_fallback),
        },
    ]
)

experiment_results.sort_values("precision_at_3", ascending=False)

,model,description,precision_at_3
3,personal_history_weighted,Персональная история пользователя с весами соб...,0.01
4,strong_history,История пользователя только по корзинам и поку...,0.00
0,popular_transactions,Топ товаров по покупкам в train,0.00
2,weighted_popularity,Глобальная популярность с весами событий,0.00
1,popular_views,Топ товаров по просмотрам и всем событиям,0.00


Лучшее качество среди проверенных простых подходов показала модель **`personal_history_weighted`**.

Идея модели:

- просмотр товара — слабый сигнал, вес 1;
- добавление в корзину — более сильный сигнал, вес 3;
- покупка — самый сильный сигнал, вес 10;
- для каждого пользователя сортируем товары по суммарному весу;
- если пользователь новый или у него мало истории, дополняем рекомендации глобально популярными товарами.

Итоговое значение на test: **`Precision@3 = 0.006584`**.

## Задание 2.3. Подготовка сервиса и Docker

Для проекта подготовлены файлы:

- `scripts/train_recommender.py` — обучение, эксперименты и сохранение артефакта;
- `artifacts/recommender_artifacts.json` — готовые рекомендации, fallback и метрики;
- `app/main.py` — FastAPI-сервис;
- `Dockerfile` — сборка API-сервиса;
- `README.md` — документация по данным, обучению, API и Docker;
- `presentation_manager.pdf` — презентация.

In [1]:
from pathlib import Path
import json

artifact_path = Path("artifacts/recommender_artifacts.json")
artifact = json.loads(artifact_path.read_text(encoding="utf-8"))

print("Модель:", artifact["model_type"])
print("Precision@3:", artifact["metrics"]["precision_at_3"])
print("Пользователей в артефакте:", len(artifact["user_recommendations"]))
print("Fallback товаров:", len(artifact["fallback_recommendations"]))
print("Первое свойство вне топ-20:", artifact["first_property_outside_top20"])

Модель: personal_history_weighted
Precision@3: 0.006584
Пользователей в артефакте: 1407580
Fallback товаров: 200
Первое свойство вне топ-20: 348
